# Step 3: Load, check and clean the data

**Rule of this notebook: the raw files in `data-pack/data/` are only ever read. Cleaned copies go to `data-clean/`.**

Run the cells top to bottom (Shift+Enter). Each section says what it does, why, and how to confirm it worked.
Requires: `pip install pandas` and a Jupyter kernel (VS Code Jupyter extension).

## 3.1 Setup and load (everything as plain text)

Why text: if pandas guesses types, the string `NA` silently becomes "missing" and `01-04-2024` may be read month-first.
We read everything as text, then convert on purpose.

In [ ]:
import os, re
import pandas as pd

RAW = os.path.join('..', 'data-pack', 'data')   # read-only
OUT = os.path.join('..', 'data-clean')          # we write here
os.makedirs(OUT, exist_ok=True)

LOG = []   # one line per fix: what, where, how many, what we did
def log(file, column, issue, rows, action):
    LOG.append(dict(file=file, column=column, issue=issue, rows=int(rows), action=action))

def load(name):
    df = pd.read_csv(os.path.join(RAW, name + '.csv'), dtype=str, keep_default_na=False)
    return df.apply(lambda s: s.str.strip())    # remove leading/trailing spaces

names = ['sku_master','stockist_master','batch_master','primary_sales','expiry_claims','cnf_stock_snapshot','claim_emails']
raw = {n: load(n) for n in names}
for n, df in raw.items():
    print(f'{n:22s} rows={len(df):>6}  cols={len(df.columns)}')

**Check:** you should see `sku_master 36`, `stockist_master 156`, `batch_master 663`, `primary_sales 93809`, `expiry_claims 4526`, `cnf_stock_snapshot 5015`, `claim_emails 30`.

## 3.2 Look for problems (nothing is changed here)

Blank cells per column, and the *shape* of each value (9 = digit, A = capital, a = lowercase). A column with several shapes is a warning sign.

In [ ]:
def shape(s): return re.sub(r'[a-z]','a',re.sub(r'[A-Z]','A',re.sub(r'\d','9',s)))

for n, df in raw.items():
    print('\n==', n)
    blanks = {c:int((df[c]=='').sum()) for c in df.columns if (df[c]=='').sum()}
    print('  blank cells:', blanks)
    for c in df.columns:
        pc = df[c][df[c]!=''].map(shape).value_counts()
        if 1 < len(pc) <= 6 and df[c].nunique() > 12:
            print(f'  {c}: shapes', pc.to_dict())

**Check:** you should see the date columns with 2-3 shapes, `stockist_code` with `AAA9999` / `aaa9999`, `sku_code` in claims with `AAA-9999`, `cnf_code` in the snapshot with `CFE01`-style codes, and `scheme_code` with `NA` / `-` / blank.

## 3.3 Fix: helper functions

Small, reusable pieces. Read them once; they are used in the cells below.

In [ ]:
def to_date(s, f, c):
    '''Parse the three date formats in these files. Stops with an error if any value fits none.'''
    iso = s.str.fullmatch(r'\d{4}-\d{2}-\d{2}')            # 2024-04-01
    dmy = s.str.fullmatch(r'\d{2}-\d{2}-\d{4}')            # 01-04-2024 (day first)
    mon = s.str.fullmatch(r'\d{2}-[A-Za-z]{3}-\d{4}')       # 01-Apr-2024
    blank = s == ''
    bad = ~(iso | dmy | mon | blank)
    if bad.any():
        raise ValueError(f'{f}.{c}: unparsed dates {s[bad].unique()[:5]}')
    out = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
    out[iso] = pd.to_datetime(s[iso], format='%Y-%m-%d')
    out[dmy] = pd.to_datetime(s[dmy], format='%d-%m-%Y')
    out[mon] = pd.to_datetime(s[mon], format='%d-%b-%Y')
    n = int((iso | mon).sum())
    if n: log(f, c, 'mixed date formats (ISO / dd-Mon-yyyy / dd-mm-yyyy)', n, 'parsed to one date type')
    return out

def num(s):
    return pd.to_numeric(s.str.replace(',', '', regex=False), errors='coerce')

def fix_codes(df, f):
    if 'stockist_code' in df:
        n = int((df.stockist_code != df.stockist_code.str.upper()).sum())
        df['stockist_code'] = df.stockist_code.str.upper()
        if n: log(f, 'stockist_code', 'lowercase code', n, 'uppercased (spaces were stripped on load)')
    if 'sku_code' in df:
        n = int(df.sku_code.str.contains('-').sum())
        df['sku_code'] = df.sku_code.str.upper().str.replace('-', '', regex=False)
        if n: log(f, 'sku_code', 'hyphen form PFL-1301', n, 'hyphen removed')
    if 'batch_no' in df:
        n = int((df.batch_no != df.batch_no.str.upper()).sum())
        df['batch_no'] = df.batch_no.str.upper()
        if n: log(f, 'batch_no', 'lowercase batch', n, 'uppercased')
    if 'cnf_code' in df:
        m = ~df.cnf_code.str.contains('-') & (df.cnf_code != '')
        df.loc[m, 'cnf_code'] = df.loc[m, 'cnf_code'].str.replace(r'^CF(?=[A-Z]\d)', 'CF-', regex=True)
        if m.any(): log(f, 'cnf_code', 'CFE01 instead of CF-E01', m.sum(), 'hyphen inserted')
    return df

def drop_exact_dups(df, f, value_col=None):
    '''Drop rows identical in EVERY field (run AFTER normalising codes). Dropped rows are saved.'''
    d = df[df.duplicated(keep='first')]
    if len(d):
        val = float(d[value_col].sum()) if value_col else float('nan')
        log(f, '(row)', f'exact duplicate rows after normalisation; value INR {val:,.0f}', len(d), 'dropped (kept first); see dropped_duplicates_*.csv')
        d.to_csv(os.path.join(OUT, f'dropped_duplicates_{f}.csv'), index=False)
    return df.drop_duplicates(keep='first').reset_index(drop=True)

## 3.3a Masters (SKU, stockist, batch)

In [ ]:
sku = raw['sku_master'].copy()
for c in ['shelf_life_months', 'pts_inr', 'mrp_inr']: sku[c] = num(sku[c])

stk = raw['stockist_master'].copy()
stk['appointed_on'] = to_date(stk.appointed_on, 'stockist_master', 'appointed_on')
log('stockist_master', 'appointed_on', 'blank (never filled)', stk.appointed_on.isna().sum(), 'left blank')
log('stockist_master', 'city', 'blank', (stk.city == '').sum(), 'left blank')

bat = fix_codes(raw['batch_master'].copy(), 'batch_master')
for c in ['mfg_date', 'expiry_date']: bat[c] = to_date(bat[c], 'batch_master', c)
bat['batch_qty_packs'] = num(bat.batch_qty_packs)
log('batch_master', 'mfg_site', 'blank (all others are GOA-01)', (bat.mfg_site == '').sum(), 'left blank')
print(sku.shape, stk.shape, bat.shape)

## 3.3b Primary sales

- three spellings of "no scheme" (blank, `NA`, `-`) become `NO_SCHEME`
- exact duplicates dropped **after** codes are standardised
- flags added, nothing deleted or filled: cancelled lines, missing order quantity, same-key lines that are not exact copies

In [ ]:
ps = fix_codes(raw['primary_sales'].copy(), 'primary_sales')
ps['invoice_date'] = to_date(ps.invoice_date, 'primary_sales', 'invoice_date')
commas = int(ps.net_value_inr.str.contains(',').sum())
for c in ['ordered_qty', 'invoiced_qty', 'free_qty', 'pts_inr', 'net_value_inr']: ps[c] = num(ps[c])
log('primary_sales', 'net_value_inr', 'numbers stored as text with commas (137,112.00)', commas, 'converted to numbers')

no = ps.scheme_code.isin(['', 'NA', '-'])
log('primary_sales', 'scheme_code', 'no scheme written as blank / NA / -', no.sum(), "set to 'NO_SCHEME'")
ps.loc[no, 'scheme_code'] = 'NO_SCHEME'

ps = drop_exact_dups(ps, 'primary_sales', 'net_value_inr')

ps['is_cancelled_line'] = ps.invoiced_qty == 0
ps['ordered_missing'] = ps.ordered_qty.isna()
ps['dup_key_conflict'] = ps.duplicated(['invoice_no','sku_code','batch_no'], keep=False) & ~ps.is_cancelled_line
log('primary_sales', 'invoiced_qty', 'zero-invoiced line, no batch (cancelled)', ps.is_cancelled_line.sum(), 'kept; flagged is_cancelled_line')
log('primary_sales', 'ordered_qty', 'blank (not recorded)', ps.ordered_missing.sum(), 'left blank; flagged ordered_missing; excluded from R11')
log('primary_sales', 'invoice_no+sku+batch', 'same key on several non-cancelled lines (not exact copies)', ps.dup_key_conflict.sum(), 'kept; flagged; examine in 3.4')
print(ps.shape)

## 3.3c Expiry claims

In [ ]:
ec = fix_codes(raw['expiry_claims'].copy(), 'expiry_claims')
m = ec.original_invoice_ref.str.startswith('INV-')
ec.loc[m, 'original_invoice_ref'] = ec.loc[m, 'original_invoice_ref'].str.replace('INV-', 'INV/', n=1, regex=False)
if m.any(): log('expiry_claims', 'original_invoice_ref', 'INV-S02/.. instead of INV/S02/..', m.sum(), 'standardised to INV/')

for c in ['claim_date', 'expiry_date_as_claimed', 'credit_note_date']: ec[c] = to_date(ec[c], 'expiry_claims', c)
commas = int(ec.credit_value_inr.str.contains(',').sum())
for c in ['qty_claimed', 'claim_value_inr', 'credit_value_inr']: ec[c] = num(ec[c])
log('expiry_claims', 'credit_value_inr', 'numbers stored as text with commas', commas, 'converted to numbers')
log('expiry_claims', 'claim_value_inr', 'blank', ec.claim_value_inr.isna().sum(), 'left blank (not derived; R5 decision)')

ec = drop_exact_dups(ec, 'expiry_claims', 'claim_value_inr')
ec['invoice_ref_missing'] = ec.original_invoice_ref == ''
print(ec.shape)

## 3.3d Stock snapshot and emails

Rows with the same date, C&F, SKU and batch but different `qty_on_hand_packs` are **both kept and flagged**. We do not add or drop them until the client explains them.

In [ ]:
sn = fix_codes(raw['cnf_stock_snapshot'].copy(), 'cnf_stock_snapshot')
sn['snapshot_date'] = to_date(sn.snapshot_date, 'cnf_stock_snapshot', 'snapshot_date')
sn['expiry_date'] = to_date(sn.expiry_date, 'cnf_stock_snapshot', 'expiry_date')
commas = int(sn.qty_on_hand_packs.str.contains(',').sum())
for c in ['qty_on_hand_packs', 'avg_monthly_outward_packs']: sn[c] = num(sn[c])
log('cnf_stock_snapshot', 'qty_on_hand_packs', 'numbers stored as text with commas (e.g. 1,234)', commas, 'converted to numbers')
sn = drop_exact_dups(sn, 'cnf_stock_snapshot')

sn['is_stockout_row'] = (sn.batch_no == '') & (sn.qty_on_hand_packs == 0)
has_batch = sn.batch_no != ''
sn['qty_conflict'] = has_batch & sn.duplicated(['snapshot_date','cnf_code','sku_code','batch_no'], keep=False)
log('cnf_stock_snapshot', 'batch_no+expiry_date', 'blank batch with qty 0 (probable stock-out marker; inference)', sn.is_stockout_row.sum(), 'kept; flagged is_stockout_row')
log('cnf_stock_snapshot', 'qty_on_hand_packs', 'same date/C&F/SKU/batch with different qty', sn.qty_conflict.sum(), 'BOTH kept, flagged qty_conflict; not summed; ask client')

em = raw['claim_emails'].copy()
em['received_date'] = to_date(em.received_date, 'claim_emails', 'received_date')
log('claim_emails', 'stockist_code_by_mailbox_rule', 'blank (sender not matched)', (em.stockist_code_by_mailbox_rule == '').sum(), 'left blank')
print(sn.shape, em.shape)

## 3.3e Save the cleaned files and the cleaning log

In [ ]:
clean = dict(sku_master=sku, stockist_master=stk, batch_master=bat, primary_sales=ps,
             expiry_claims=ec, cnf_stock_snapshot=sn, claim_emails=em)
for n, df in clean.items():
    df.to_csv(os.path.join(OUT, n + '.csv'), index=False)
log_df = pd.DataFrame(LOG)
log_df.to_csv(os.path.join(OUT, 'cleaning_log.csv'), index=False)
log_df

**Check:** the log should list about 28 lines. Open `data-clean/cleaning_log.csv` and read it; this becomes the data-quality log in 3.5.

## 3.4 Validate: did the cleaning do what we said?

Every line below is an `assert`. If one fails, Jupyter stops with an error. All passing means the cleaned files are consistent with our rules.

In [ ]:
# row counts: raw minus exact duplicates dropped
assert len(ps) == 93809 - 746,  len(ps)
assert len(ec) == 4526 - 23,    len(ec)
assert len(sn) == 5015 and len(bat) == 663 and len(sku) == 36 and len(stk) == 156 and len(em) == 30

# dates are real dates now, not text
for df, cols in [(ps,['invoice_date']), (ec,['claim_date','expiry_date_as_claimed','credit_note_date']),
                 (bat,['mfg_date','expiry_date']), (sn,['snapshot_date','expiry_date'])]:
    for c in cols: assert str(df[c].dtype).startswith('datetime64'), (c, df[c].dtype)

# codes
assert not ps.stockist_code.str.contains(r'\s|[a-z]').any() and not ec.stockist_code.str.contains(r'\s|[a-z]').any()
assert not ec.sku_code.str.contains('-').any()
assert set(sn.cnf_code) == {'CF-E01','CF-N01','CF-N02','CF-S01','CF-S02','CF-W01'}, set(sn.cnf_code)
assert not ec.batch_no.str.contains('[a-z]').any()
assert not ec.original_invoice_ref.str.startswith('INV-').any()

# scheme and numbers
assert not ps.scheme_code.isin(['', 'NA', '-']).any()
assert ec.credit_value_inr.dtype.kind == 'f' and ps.net_value_inr.dtype.kind == 'f' and sn.qty_on_hand_packs.dtype.kind == 'f'

# no exact duplicates remain, and no rows lost or invented beyond the log
assert not ps.drop(columns=['is_cancelled_line','ordered_missing','dup_key_conflict']).duplicated().any()
assert not ec.drop(columns=['invoice_ref_missing']).duplicated().any()

# money did not change except for the dropped duplicates
raw_net = num(raw['primary_sales'].net_value_inr).sum()
print('primary net value raw       :', f'{raw_net:,.0f}')
print('primary net value clean     :', f'{ps.net_value_inr.sum():,.0f}')
print('difference (= dropped dups) :', f'{raw_net - ps.net_value_inr.sum():,.0f}')
print('NO_SCHEME lines             :', int((ps.scheme_code == 'NO_SCHEME').sum()))
print('ALL CHECKS PASSED')

**Check:** `ALL CHECKS PASSED` prints. The difference line should be about 40.7 million INR (about 4.07 crore), the value of the dropped duplicate lines, and it should equal the sum in `dropped_duplicates_primary_sales.csv`.

Next: 3.4 link checks (do the codes in one file exist in the others?), then 3.5 data dictionary and data-quality log.